# Causal Inference: Confounding, Counterfactuals, and Causal Graphs


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/14-Causal-Inference.ipynb)

So far in this course we have built tools that are very good at answering one kind of
question: **given what I observe, what should I predict?** Clustering, regression, and
classification all learn associations in data.

But many of the questions we actually care about as data scientists are of a different
kind. They are questions about **what would happen if we acted**:

* If we *change* the checkout button from blue to green, will more people buy?
* If a patient *takes* this drug, will they recover faster?
* If a student *enrolls* in the tutoring program, will their grade go up?
* If the city *raises* the minimum wage, will employment fall?


These are **causal** questions, and — as we will see — no amount of predictive accuracy
answers them by itself.



# A fitted regression is not a causal claim

## You just fit regressions. What does a coefficient mean?

In Lectures 10 and 11 you fit models like

```python
smf.ols("Lottery ~ Literacy + Wealth + Region", data=df).fit().summary()
```

and read off a coefficient, a confidence interval, and a $p$-value for each variable.

There are **two very different readings** of a coefficient $\hat\beta_1$ on $X$:

**Associational reading (what OLS estimates)**

"Among the units in my data, those with $X$ one unit higher have $Y$ higher by
$\hat\beta_1$ on average."

**Causal reading (what a decision-maker wants)**

"If I *raised* $X$ by one unit, $Y$ would *change* by $\hat\beta_1$ on average."


Least squares gives you the first reading, always. It gives you the second **only under
extra assumptions that the data cannot check for you.** This lecture is about what those
assumptions are, and what happens when they fail.


## A regression you would recognize

Two variables measured daily across the summer: **ice cream sales** and **drowning deaths**.
Let's fit the regression exactly as we did in Lectures 10 and 11.

In [ ]:
#| echo: true
import numpy as np, pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(701)
n = 200
temp = rng.uniform(60, 100, n)                        # daily high temperature (°F)
ice_cream = 2.0 * temp + rng.normal(0, 15, n)         # ice cream sales
drownings = 0.10 * temp + rng.normal(0, 2.0, n)       # drowning deaths
# NOTE: ice cream does NOT appear in the equation for drownings.

df = pd.DataFrame({"ice_cream": ice_cream, "drownings": drownings, "temp": temp})
res = smf.ols("drownings ~ ice_cream", data=df).fit()
print(res.summary().tables[1])

A positive slope, a tight confidence interval, $p \approx 0$. Every diagnostic from
Lecture 10 says the fit is fine.
**Should the city ban ice cream to reduce drownings?**


## Correlation is not causation

* The regression is *correct* as a description of the data: on days with more ice cream
  sales there **are** more drownings.
* But look at how the data were generated: a third variable, **hot weather**, drives both.
  Hot days increase ice cream sales *and* send more people swimming.
* Temperature is a **confounder**. Ice cream has **no** effect on drowning, and banning it
  would change nothing.


The slogan "correlation is not causation" is famous, and yet the mistakes it warns about
are everywhere — most often hidden inside a regression table that *looks* authoritative.


## The picture behind the slogan

In [ ]:
#| echo: false
#| fig-align: center
import networkx as nx
import matplotlib.pyplot as plt

def draw_fork(ax, z, x, y, title):
    """Common cause z -> x, z -> y (the causal-graph style used later in this
    lecture), plus the association we *see* between x and y, dashed and crossed out."""
    pos = {z: (1, 1), x: (0, 0), y: (2, 0)}
    G = nx.DiGraph([(z, x), (z, y)])
    nx.draw(G, pos, ax=ax, with_labels=True, node_color="#cde", node_size=3200,
            font_size=10, arrowsize=22, edgecolors="#333", linewidths=1.2)
    nx.draw_networkx_edges(nx.DiGraph([(x, y)]), pos, ax=ax, node_size=3200,
                           style="dashed", edge_color="#c0392b", width=2, arrowsize=22)
    ax.text(1, 0, "✗", color="#c0392b", fontsize=22, ha="center", va="center",
            bbox=dict(boxstyle="circle,pad=0.1", fc="white", ec="none"))
    ax.text(1, -0.22, "association, not cause", color="#c0392b",
            fontsize=9, ha="center", va="top")
    ax.set_xlim(-0.6, 2.6); ax.set_ylim(-0.5, 1.4)
    ax.set_title(title, fontsize=11)

fig, ax = plt.subplots(figsize=(5.5, 3.4))
draw_fork(ax, "Temperature", "Ice cream\nsales", "Drownings",
          "Temperature is a common cause of both")
plt.tight_layout(); plt.show()

Both arrows point *out of* temperature. Follow them and you can get from ice cream to
drownings without either one causing the other. Later in this lecture this shape gets a
name and a set of rules; for now, the shape is the whole story.


## Confounding

**Definition**

A **confounder** is a variable that causally affects **both** the treatment (or explanatory
variable) $X$ **and** the outcome $Y$. It creates an association between $X$ and $Y$ that
is not due to $X$ causing $Y$.


Multiple regression can *sometimes* fix this — **if** we measure the confounder and include it:

In [ ]:
#| echo: true
naive = smf.ols("drownings ~ ice_cream", data=df).fit().params["ice_cream"]
adj   = smf.ols("drownings ~ ice_cream + temp", data=df).fit().params["ice_cream"]
print(f"slope on ice_cream, ignoring temp:  {naive:+.3f}")
print(f"slope on ice_cream, adjusting temp: {adj:+.3f}   (truth: 0)")

Adding `temp` to the formula drives the ice-cream coefficient to zero. So "controlling for"
a variable is just adding it to the regression — but **which** variables to add is a causal
question that the fitting procedure cannot answer. If temperature had not been measured, no
amount of model selection would have saved us.


## Same trap, higher stakes

A hospital analytics team fits a model on patient records and finds a strong,
statistically significant relationship:

> Patients who **visit the hospital** are *more* likely to die within the year than
> patients who do not.

Should we conclude that hospitals *cause* death and recommend people avoid them?


Of course not. Sick people go to hospitals. The hospital visit is **associated** with
death because both are driven by a common cause — being sick. Same shape as the ice cream
picture, with the names changed.

In [ ]:
#| echo: false
#| fig-align: center
fig, ax = plt.subplots(figsize=(4.4, 2.7))
draw_fork(ax, "Sickness", "Hospital\nvisit", "Death",
          "Sickness is a common cause of both")
plt.tight_layout(); plt.show()

The model is a perfectly good *predictor* ("I see a hospital visit, I predict higher
mortality") and a catastrophically bad *guide to action* ("close the hospitals"). That
gap — **prediction** vs. **intervention** — is the subject of this lecture
[@angrist2009mostly].


# Prediction vs. intervention

## Two different questions

It is worth being precise about the distinction.


**Prediction (association)**

$$
P(Y \mid X = x)
$$

"Among people I *observe* to have $X = x$, what is the distribution of $Y$?"

This is what supervised learning — including every regression you have fit — estimates.

**Intervention (causation)**

$$
P(Y \mid do(X = x))
$$

"If I *set* $X = x$ for everyone, what is the distribution of $Y$?"

This is what a decision-maker needs. [@pearl2009causality]


In general $P(Y \mid X=x) \neq P(Y \mid do(X=x))$. The whole game is figuring out when —
and how — we can recover the second from data that only shows us the first.


## Where do spurious associations come from?

A correlation between $X$ and $Y$ can arise for several reasons — only one of which is
"$X$ causes $Y$":

* **$X$ causes $Y$** (the thing we want),
* **$Y$ causes $X$** (reverse causation),
* **a confounder $Z$ causes both** $X$ and $Y$ (ice cream / drowning),
* **collider bias** — conditioning or selecting on a *common effect* of $X$ and $Y$
  (later in this lecture),
* **pure chance** — with enough variables, some will correlate by luck.


# Potential outcomes

## The potential outcomes framework

To reason precisely, we use the **Neyman–Rubin potential outcomes** model
[@splawa1990application; @rubin1974estimating]. Our running example: does an **optional
tutoring program** raise students' **exam scores**? Students choose whether to sign up.

For each unit $i$ (here, a student) and a binary **treatment** $T_i \in \{0,1\}$ (tutored
or not):

* $Y_i(1)$ — the exam score we *would* see if student $i$ were tutored,
* $Y_i(0)$ — the exam score we *would* see if student $i$ were **not** tutored.


The **individual treatment effect** is

$$
\tau_i = Y_i(1) - Y_i(0).
$$


These are called *potential* outcomes because, for any given unit, only **one** of them
is ever realized.


## If we could see everything (we can't)

Imagine an all-seeing observer who knows **both** potential outcomes for six students.
The three weaker students signed up for tutoring; the three stronger ones did not.

| Student | $T_i$ | $Y_i(0)$ | $Y_i(1)$ | $\tau_i$ |
|:-------:|:-----:|:--------:|:--------:|:--------:|
| Ann  | 1 | 60 | 62 | 2 |
| Bob  | 1 | 65 | 67 | 2 |
| Cara | 1 | 70 | 72 | 2 |
| Dan  | 0 | 68 | 70 | 2 |
| Eve  | 0 | 70 | 72 | 2 |
| Finn | 0 | 72 | 74 | 2 |

* Tutoring raises **every** student's score by exactly 2 points, so the average effect
  is 2.
* Yet compare the scores we would *actually observe*: tutored students average
  $(62 + 67 + 72)/3 = 67$; untutored students average $(68 + 70 + 72)/3 = 70$.
  The naive comparison says tutoring **lowers** scores by 3.
* The gap is in the $Y_i(0)$ column: those who chose tutoring would have scored $65$ on
  average without it, versus $70$ for those who didn't. That $-5$ is **not** an effect of
  tutoring — it is who signed up.


## The fundamental problem of causal inference

Nobody is all-seeing. We only ever observe the outcome for the treatment actually received:

$$
Y_i^{\text{obs}} = T_i\, Y_i(1) + (1 - T_i)\, Y_i(0).
$$

| Student | $T_i$ | $Y_i(0)$ | $Y_i(1)$ | $\tau_i$ |
|:-------:|:-----:|:--------:|:--------:|:--------:|
| Ann  | 1 | [?]{style="color:#b0b0b0"} | 62 | [?]{style="color:#b0b0b0"} |
| Bob  | 1 | [?]{style="color:#b0b0b0"} | 67 | [?]{style="color:#b0b0b0"} |
| Cara | 1 | [?]{style="color:#b0b0b0"} | 72 | [?]{style="color:#b0b0b0"} |
| Dan  | 0 | 68 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |
| Eve  | 0 | 70 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |
| Finn | 0 | 72 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |

The same table, as the world shows it to us. The other potential outcome — the
**counterfactual** — is missing, and every $\tau_i$ needs a grey cell.


We can never compute $\tau_i$ for a single individual. Causal inference is fundamentally
a **missing data problem** [@imbens2015causal]. (The equation also quietly assumes each
student's score depends only on their *own* treatment, not on anyone else's.)


## The Average Treatment Effect (ATE)

Since individual effects are unknowable, we target **population averages**.

**Definition**

The **Average Treatment Effect** is the expected difference between a unit's two potential
outcomes, averaged over the whole population:

$$
\text{ATE} = \mathbb{E}[\,Y(1) - Y(0)\,] = \mathbb{E}[Y(1)] - \mathbb{E}[Y(0)].
$$


On the tutoring table, $\text{ATE} = 2$. A related quantity you will meet:
the **ATT**, the effect *on the treated*, $\mathbb{E}[Y(1) - Y(0)\mid T=1]$. It answers
"did the program help the people we actually enrolled?" — the question a program
manager is asked.


## Why we can't just compare treated vs. untreated

The tempting estimator is the **naive difference in means**:

$$
\underbrace{\mathbb{E}[Y \mid T=1] - \mathbb{E}[Y \mid T=0]}_{\text{what we can measure}}.
$$

Add and subtract the missing counterfactual $\mathbb{E}[Y(0)\mid T=1]$ to decompose it:

$$
\underbrace{\mathbb{E}[Y(1)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]}_{\text{naive difference}}
= \underbrace{\text{ATT}}_{\text{causal}}
+ \underbrace{\mathbb{E}[Y(0)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]}_{\text{selection bias}}.
$$


Check it against the all-seeing table: $\text{ATT} = 2$, selection bias $= 65 - 70 = -5$,
and $2 + (-5) = -3$ — exactly the naive difference. The arithmetic was right; the causal
reading was wrong.


## Selection bias

**Definition**

**Selection bias** is the difference in outcomes that treated and untreated units would
have shown *even without any treatment*: $\mathbb{E}[Y(0)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]$.
It is nonzero whenever *who gets treated* depends on things that also affect the outcome.


* Exactly the hospital example: patients who visit are sicker to begin with. And exactly
  the tutoring table: the students who signed up were weaker to begin with.
* The coefficient on `T` in the regression `Y ~ T` *is* the naive difference — so it
  inherits the same bias. Our job is to make that term vanish.


**A note on terminology**

We use *selection bias* in the sense above — a baseline difference between those who were
and were not treated (the econometrics usage). Epidemiologists reserve the term for bias
caused by how units get *into the sample*. Later in this lecture we call that **collider
bias** and treat it as a separate mechanism; keep the two apart.


# Simpson's paradox

## Confounding in numbers: a kidney stone study

A famous real study compared two treatments for kidney stones
[@charig1986comparison]. Here are the recovery rates:

| | Treatment A (surgery) | Treatment B (less invasive) |
|:--|:--:|:--:|
| **Small stones** | 81/87 = **93%** | 234/270 = **87%** |
| **Large stones** | 192/263 = **73%** | 55/80 = **69%** |
| **Overall** | 273/350 = **78%** | 289/350 = **83%** |

The same table as data, which the next two slides use:

In [ ]:
#| echo: true
# Recovery counts (successes, total) by treatment and stone size
data = {
    "A": {"small": (81, 87),  "large": (192, 263)},
    "B": {"small": (234, 270), "large": (55, 80)},
}

for tx in ("A", "B"):
    s_succ, s_tot = data[tx]["small"]
    l_succ, l_tot = data[tx]["large"]
    overall = (s_succ + l_succ) / (s_tot + l_tot)
    print(f"Treatment {tx}: small={s_succ/s_tot:.0%}, "
          f"large={l_succ/l_tot:.0%}, overall={overall:.0%}")

Treatment A wins for **small** stones **and** for **large** stones — yet Treatment B wins
**overall**.


**Stone size is a confounder.** Doctors gave the risky Treatment A to the *hard* (large-stone)
cases and Treatment B to the *easy* (small-stone) cases. The overall numbers mix effect
with case-mix.


## The two fingerprints of a confounder

In [ ]:
#| echo: false
#| fig-align: center
import matplotlib.pyplot as plt

colors = {"A": "#2c7fb8", "B": "#d95f02"}
strata = ["small", "large"]
rate = {tx: {z: data[tx][z][0] / data[tx][z][1] for z in strata} for tx in "AB"}
n    = {tx: {z: data[tx][z][1] for z in strata} for tx in "AB"}
for tx in "AB":
    n[tx]["overall"] = sum(data[tx][z][1] for z in strata)
    rate[tx]["overall"] = sum(data[tx][z][0] for z in strata) / n[tx]["overall"]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 3.4),
                               gridspec_kw={"width_ratios": [2.2, 1]})

# Left: recovery rate by stratum, bar width proportional to number of patients
groups = ["small", "large", "overall"]
gap = 0.03
for i, z in enumerate(groups):
    for tx, side in (("A", -1), ("B", +1)):
        w = 0.42 * n[tx][z] / 350
        left = i - gap - w if side < 0 else i + gap
        ax0.bar(left, rate[tx][z], width=w, align="edge", color=colors[tx],
                label=f"Treatment {tx}" if i == 0 else None)
        ax0.text(left + w / 2, rate[tx][z] + 0.02, f"{rate[tx][z]:.0%}",
                 ha="center", va="bottom", fontsize=9)
        ax0.text(left + w / 2, 0.04, f"n={n[tx][z]}", ha="center", va="bottom",
                 fontsize=8, color="white", rotation=90 if w < 0.2 else 0)
ax0.set_xticks(range(3), ["Small stones", "Large stones", "Overall"])
ax0.set_ylim(0, 1.25)
ax0.set_ylabel("Recovery rate")
ax0.set_title("Recovery rate (bar width ∝ number of patients)")
ax0.legend(loc="upper center", ncol=2, fontsize=9, frameon=False)

# Right: case mix of each treatment arm (100% stacked)
share = {tx: {z: n[tx][z] / n[tx]["overall"] for z in strata} for tx in "AB"}
for j, tx in enumerate("AB"):
    ax1.bar(j, share[tx]["small"], color="#bbbbbb", label="small stones" if j == 0 else None)
    ax1.bar(j, share[tx]["large"], bottom=share[tx]["small"], color="#555555",
            label="large stones" if j == 0 else None)
    ax1.text(j, share[tx]["small"] / 2, f"{share[tx]['small']:.0%} small",
             ha="center", va="center", fontsize=9)
    ax1.text(j, share[tx]["small"] + share[tx]["large"] / 2, f"{share[tx]['large']:.0%} large",
             ha="center", va="center", fontsize=9, color="white")
ax1.set_xticks([0, 1], ["Treatment A", "Treatment B"])
ax1.set_ylim(0, 1)
ax1.set_yticks([0, 0.5, 1], ["0%", "50%", "100%"])
ax1.set_title("Case mix of each treatment arm")

plt.tight_layout()
plt.show()

Every Simpson's table carries **two fingerprints** of the confounder, both visible here:

1. **The case mix differs between the arms** — 75% of A's patients had large stones vs.
   23% of B's. Those two numbers are the ones to hunt for in any such table; they are why
   A's overall rate is dragged toward its hard cases and B's toward its easy ones.
2. **The strata differ in outcome** — large stones recover less often under *either*
   treatment.


## Give both arms the same case mix

The overall row compared A *on mostly large stones* with B *on mostly small stones*. The fair
question: **what would each treatment's recovery rate be if it had been given to the whole
study population** — the 51% small / 49% large mix we actually have?

In [ ]:
#| echo: true
n_small = data["A"]["small"][1] + data["B"]["small"][1]      # 357 patients
n_large = data["A"]["large"][1] + data["B"]["large"][1]      # 343 patients
p_small, p_large = n_small / (n_small + n_large), n_large / (n_small + n_large)
print(f"P(small) = {p_small:.2f}, P(large) = {p_large:.2f}")

for tx in ("A", "B"):
    rate = {z: data[tx][z][0] / data[tx][z][1] for z in ("small", "large")}
    adjusted = rate["small"] * p_small + rate["large"] * p_large
    print(f"Treatment {tx}: adjusted recovery = {adjusted:.0%}")

Adjusted, **A ≈ 83% beats B ≈ 78%** — the crude 78% vs. 83% flips back to agree with the
strata. This is the **adjustment formula**, also called **direct standardization**:

$$
P(Y \mid do(T{=}t)) \;=\; \sum_z P(Y \mid T{=}t,\, Z{=}z)\; P(Z{=}z),
$$

each arm's within-stratum rates weighted by the **population's** stratum shares, not the
arm's own. The regression `Y ~ T + Z` is the same move in model form — compare within
strata, then average — which is why adding `temp` fixed the ice cream slope.


## Simpson's paradox

**Definition**

**Simpson's paradox** occurs when an association between $X$ and $Y$ that holds in
**every subgroup** of the data **reverses** (or disappears) when the subgroups are
combined. It happens when a third variable $Z$ is distributed differently across the
levels of $X$ *and* is itself related to $Y$ — the two fingerprints.


* The **within-stratum** comparisons hold $Z$ fixed. When $Z$ is a confounder — as stone
  size is here — they are the ones to trust, and standardizing just re-weights them.
  **Treatment A is better.**
* The **overall** comparison is contaminated by selection bias: A's patients were sicker
  to begin with.


The paradox is not a mathematical curiosity — it is a warning: **aggregated associations
can point the opposite direction from the truth.** But "always adjust" is not the lesson
either. Deciding *which* variables to adjust for is itself a causal question, and causal
graphs (later in this lecture) are how we answer it.


# Randomization

## The gold standard: randomized experiments

How do we break confounding by design? **Randomly assign** the treatment.

If a coin flip decides $T_i$, then treatment is *statistically independent* of the
potential outcomes:

$$
\{Y_i(0), Y_i(1)\} \perp\!\!\!\perp T_i.
$$


Independence kills the selection-bias term: in expectation the two groups are identical in
every respect (measured *and* unmeasured) except the treatment:

$$
\mathbb{E}[Y(0)\mid T{=}1] = \mathbb{E}[Y(0)\mid T{=}0].
$$


It also makes the treated a representative sample of everyone, so $\text{ATT} = \text{ATE}$:
the naive difference in means is an **unbiased estimate of the ATE** [@imbens2015causal].


## Randomization removes confounding

In [ ]:
#| echo: false
#| fig-align: center
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
n = 2000
severity = rng.uniform(0, 1, n)          # a confounder: how sick each patient is

# TRUE treatment effect is +5 for everyone
def outcomes(T, severity):
    return 50 + 5 * T - 30 * severity + rng.normal(0, 3, len(T))

# Observational: sicker patients are MORE likely to get the drug
p_treat_obs = severity
T_obs = (rng.uniform(0, 1, n) < p_treat_obs).astype(int)
Y_obs = outcomes(T_obs, severity)
naive_obs = Y_obs[T_obs == 1].mean() - Y_obs[T_obs == 0].mean()

# Randomized: coin flip, independent of severity
T_rct = (rng.uniform(0, 1, n) < 0.5).astype(int)
Y_rct = outcomes(T_rct, severity)
naive_rct = Y_rct[T_rct == 1].mean() - Y_rct[T_rct == 0].mean()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.3, 1, 1]})

vals = [5, naive_obs, naive_rct]
axes[0].bar(["True effect", "Observational\n(confounded)", "Randomized\n(RCT)"], vals,
            color=["#2c7fb8", "#d95f02", "#1b9e77"])
axes[0].axhline(5, ls="--", c="gray")
axes[0].axhline(0, c="k", lw=0.8)
for i, v in enumerate(vals):
    axes[0].text(i, v + (0.3 if v >= 0 else -0.3), f"{v:+.1f}",
                 ha="center", va="bottom" if v >= 0 else "top")
axes[0].set_ylim(min(vals) - 1.5, max(vals) + 1.5)
axes[0].set_ylabel("Estimated treatment effect")
axes[0].set_title("Naive difference in means", fontsize=10)

bins = np.linspace(0, 1, 11)
for ax, T, col, title in ((axes[1], T_obs, "#d95f02", "Observational: who got the drug?"),
                          (axes[2], T_rct, "#1b9e77", "Randomized: who got the drug?")):
    ax.hist(severity[T == 0], bins, alpha=0.55, color="#888888", label="untreated")
    ax.hist(severity[T == 1], bins, alpha=0.55, color=col, label="treated")
    ax.set_xlabel("Severity (the confounder)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
axes[1].set_ylabel("Number of patients")

plt.tight_layout()
plt.show()

The true effect is $+5$. The **observational** estimate is badly biased, and the middle
panel shows the mechanism: the treated group is dominated by severe cases, who do worse for
reasons that have nothing to do with the drug. Under **randomization** the two severity
histograms lie on top of each other, and the naive estimate recovers the truth.

Try it yourself in the notebook: change `p_treat_obs` so that *healthier* patients are more
likely to be treated, and predict the direction of the bias before you run the cell.

When you *can* randomize, do it — clinical trials and A/B tests are randomized experiments.
But often you cannot, and the rest of this lecture is about what to do then.


# Causal graphs

## Drawing our assumptions: causal graphs

Often we **cannot** randomize: it would be unethical (randomly assign people to smoke),
impossible (re-run a minimum-wage law), too slow or expensive — or the treatment already
happened and we only have observational data.

A **causal graph** is a *directed acyclic graph* (DAG): nodes are variables, and an arrow
$X \rightarrow Y$ means "$X$ is a direct cause of $Y$" [@pearl2016primer].


The graph encodes **assumptions** we bring to the data (from domain knowledge). The data
alone cannot draw it for us — but once drawn, the graph tells us *exactly* which variables
to control for.


We will study graphs in their own right later in the course. For now you need only the
handful of terms on the next slide.


## Reading a causal graph

In [ ]:
#| echo: false
#| fig-align: center
#| label: example-dag
def draw(ax, edges, pos, title="", node_color="#cde", node_size=1400, font_size=12):
    """Lecture-wide DAG style: pale nodes, dark outlines, black arrows."""
    G = nx.DiGraph(edges)
    colors = [node_color.get(v, "#cde") for v in G] if isinstance(node_color, dict) else node_color
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=node_size, font_size=font_size, arrowsize=22,
            edgecolors="#333", linewidths=1.2)
    ax.set_title(title, fontsize=11)

example_edges = [("A\nage", "X\nexercise"), ("A\nage", "Y\nheart"),
                 ("X\nexercise", "W\nweight"), ("W\nweight", "Y\nheart"),
                 ("M\nmotivation", "X\nexercise"), ("M\nmotivation", "D\ndiet"),
                 ("D\ndiet", "Y\nheart")]
example_pos = {"A\nage": (1, -1.3), "M\nmotivation": (-0.6, 1.4), "D\ndiet": (0.7, 1.0),
               "X\nexercise": (0, 0), "W\nweight": (1, 0), "Y\nheart": (2, 0)}

fig, ax = plt.subplots(figsize=(5.4, 3.6))
draw(ax, example_edges, example_pos, node_size=2600, font_size=9)
ax.set_xlim(-1.05, 2.4); ax.set_ylim(-1.75, 1.8)
plt.tight_layout(); plt.show()

* **Node** — a variable. **Arrow** $A \to X$ — "$A$ is a direct cause of $X$" ($A$ is a
  **parent** of $X$, $X$ a **child** of $A$).
* **Directed path** — follow the arrows head-to-tail: $X \to W \to Y$. **Descendants**
  of $X$: $W$, $Y$. **Ancestors** of $X$: $A$, $M$.
* **Acyclic** — no directed path returns to its start: a **DAG**.
* **Path** — any route between two nodes, ignoring direction, no node twice:
  e.g. $X \leftarrow M \to D \to Y$.


Causal effects flow along **directed** paths; spurious associations flow along the *other*
paths when they are **open**. Every path is built from just **three building blocks**.


## The three building blocks

In [ ]:
#| echo: false
#| fig-align: center
#| label: building-blocks
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))

# Chain (mediator): X -> M -> Y
draw(axes[0], [("X", "M"), ("M", "Y")],
     {"X": (0, 0), "M": (1, 0), "Y": (2, 0)},
     "Chain / mediator\nX → M → Y")

# Fork (confounder): X <- Z -> Y
draw(axes[1], [("Z", "X"), ("Z", "Y")],
     {"Z": (1, 1), "X": (0, 0), "Y": (2, 0)},
     "Fork / confounder\nX ← Z → Y")

# Collider: X -> C <- Y
draw(axes[2], [("X", "C"), ("Y", "C")],
     {"C": (1, 0), "X": (0, 1), "Y": (2, 1)},
     "Collider\nX → C ← Y")

plt.tight_layout()
plt.show()

* **Chain** $X \to M \to Y$: $M$ is a **mediator** (it transmits the effect).
* **Fork** $X \leftarrow Z \to Y$: $Z$ is a **confounder** (common cause).
* **Collider** $X \to C \leftarrow Y$: $C$ is a **common effect**.


Any path is a sequence of these: each interior node is a mediator, a confounder, or a
collider *on that path*.


## Open and blocked paths

A path carries association between its endpoints only if it is **open**. Two rules decide:

1. A **chain** or **fork** node is *open by default* and becomes **blocked** if you condition
   on it (adjust / stratify / put it in the regression).
2. A **collider** node is *blocked by default* and becomes **open** if you condition on it
   (or on any of its descendants).


A path is **blocked** if *at least one* of its interior nodes is blocked; otherwise it is
**open**. Two variables are (conditionally) independent when **every** path between them is
blocked.


Sanity check on the example DAG: $X \leftarrow A \to Y$ is open (fork, nothing conditioned) —
the same shape as the ice-cream/temperature confounding we started with. Condition on $A$
and it closes.


## The golden rule: adjust for forks, not colliders

The three structures behave *oppositely* when you condition (adjust/stratify) on the middle node:

| Structure | Middle node | Adjust? | Why |
|:--|:--|:--:|:--|
| Fork $X \leftarrow Z \to Y$ | confounder | **Yes** | blocks the backdoor path |
| Chain $X \to M \to Y$ | mediator | **No**\* | it *is* the effect you want |
| Collider $X \to C \leftarrow Y$ | collider | **No!** | *creates* a spurious association |

: {tbl-colwidths="[36,18,12,34]"}

\* Adjust for a mediator only if you specifically want the *direct* (not total) effect.

"Just throw every variable into the regression" is **wrong** — controlling for a collider
or a mediator *introduces* bias. Choosing controls is a causal decision, not a statistical
one [@cinelli2020crash].


# The backdoor criterion

## Confounders: the backdoor

A **backdoor path** is a non-causal path from $X$ to $Y$ that starts with an arrow *into*
$X$ (e.g. $X \leftarrow Z \to Y$). Backdoor paths are the graph-language for confounding.

**Backdoor criterion:** a set of variables $S$ suffices to identify the causal effect of
$X$ on $Y$ if $S$ **blocks every backdoor path** and contains **no descendants of $X$**
[@pearl2009causality].


When such an $S$ exists, the **direct standardization** we used on the kidney stones applies
with $S$ in place of the stratifying variable:

$$
P(Y \mid do(X{=}x)) = \sum_{s} P(Y \mid X{=}x, S{=}s)\, P(S{=}s).
$$

The backdoor criterion is what tells you the formula is *legitimate*. It was legitimate
for kidney-stone size because stone size is a fork.


## The backdoor procedure, by hand

Given a DAG, a treatment $X$ and an outcome $Y$:

1. **List every path** from $X$ to $Y$ (ignore arrow direction while tracing; visit no node
   twice).
2. **Set aside the causal paths** — those where every arrow points away from $X$ toward $Y$.
   Never block these (that would be adjusting for a mediator).
3. **Classify the rest.** A path whose first arrow points *into* $X$ is a **backdoor path** —
   mark it open or blocked (forks/chains open, colliders closed). A path whose first arrow
   points *out of* $X$ but that is not causal — it runs into a collider, e.g.
   $X \to C \leftarrow Y$ — is **neither**: blocked by default, and it needs no action *as
   long as you leave $C$ and its descendants alone*.
4. **Choose $S$** that blocks every open backdoor path — a fork or chain node on each —
   containing **no collider or collider-descendant on *any* path** (backdoor or not) and
   **no descendant of $X$**.
5. **Report $S$** (there may be several valid choices; the smallest is usually best).


Then adjust for $S$ by whichever estimator you like — regression, stratification, matching,
propensity scores (later in this lecture).


## Your turn: pick the adjustment set

Tutoring ($X$) and final grade ($Y$), with:
$\text{GPA}_{\text{prior}} \to X$, $\text{GPA}_{\text{prior}} \to Y$,
$X \to \text{Hours} \to Y$, and $X \to \text{HonorRoll} \leftarrow Y$.

Which of these adjustment sets are valid for the **total** effect of $X$ on $Y$?

* (a) $\{\text{GPA}_{\text{prior}}\}$
* (b) $\{\text{GPA}_{\text{prior}}, \text{Hours}\}$
* (c) $\{\text{GPA}_{\text{prior}}, \text{HonorRoll}\}$
* (d) $\{\}$ (nothing)


Pause and decide before you read on: run the five steps.


## Your turn: the answer

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| echo: false
#| fig-align: center
#| label: tutoring-dag
from slideUtilities import OKABE_ITO

tutor_edges = [("GPA\nprior", "X"), ("GPA\nprior", "Y"), ("X", "Hours"), ("Hours", "Y"),
               ("X", "Honor\nroll"), ("Y", "Honor\nroll")]
tutor_pos = {"X": (0, 0), "GPA\nprior": (1, 1.25), "Hours": (1, 0.3),
             "Honor\nroll": (1, -0.95), "Y": (2, 0)}
roles = {"GPA\nprior": OKABE_ITO[2], "Hours": OKABE_ITO[0], "Honor\nroll": OKABE_ITO[3]}

fig, ax = plt.subplots(figsize=(4.6, 3.2))
draw(ax, tutor_edges, tutor_pos, node_color=roles, node_size=1700, font_size=9,
     title="X = tutoring, Y = final grade\n"
           "green = confounder, orange = mediator, red = collider")
ax.set_xlim(-0.3, 2.3); ax.set_ylim(-1.35, 1.6)
plt.tight_layout(); plt.show()

| Path | Type | Action |
|:--|:--|:--|
| $X \to \text{Hours} \to Y$ | causal (open) | leave alone |
| $X \leftarrow \text{GPA} \to Y$ | backdoor (open fork) | block with GPA |
| $X \to \text{Honor} \leftarrow Y$ | **neither** (blocked collider) | do not touch |

: {tbl-colwidths="[44,34,22]"}


**Only (a).** (b) blocks the mediator; (c) opens the collider; (d) leaves the backdoor
open. Restricting the study to honor-roll students — the "cleaner subset" — is (c) in
disguise.


## Colliders: adjusting can *create* bias

Conditioning on a collider (or its descendant) opens a spurious path. Classic example:
suppose **talent** and **looks** are independent in the general population, but both help an
actor get **famous**.

In [ ]:
#| echo: true
import numpy as np
rng = np.random.default_rng(701)
n = 10_000

talent = rng.normal(0, 1, n)
looks  = rng.normal(0, 1, n)          # independent of talent by construction
famous = (talent + looks + rng.normal(0, 0.5, n)) > 1.5   # collider: common effect

corr_all    = np.corrcoef(talent, looks)[0, 1]
corr_famous = np.corrcoef(talent[famous], looks[famous])[0, 1]
print(f"corr(talent, looks) overall        = {corr_all:+.2f}")
print(f"corr(talent, looks) among famous    = {corr_famous:+.2f}")

In [ ]:
#| echo: false
#| fig-align: center
#| label: collider-scatter
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.scatter(talent[~famous], looks[~famous], s=4, color="#bbb", label="not famous")
ax.scatter(talent[famous], looks[famous], s=6, color=OKABE_ITO[0], label="famous")
grid = np.linspace(-4, 4, 2)
for mask, color, name in [(np.ones(n, bool), "#555", "everyone"), (famous, OKABE_ITO[3], "famous only")]:
    slope, icpt = np.polyfit(talent[mask], looks[mask], 1)
    ax.plot(grid, icpt + slope * grid, color=color, lw=2,
            label=f"OLS line, {name} (r = {np.corrcoef(talent[mask], looks[mask])[0, 1]:+.2f})")
ax.plot(grid, 1.5 - grid, ls="--", color="black", lw=1, label="talent + looks = 1.5")
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_xlabel("talent"); ax.set_ylabel("looks")
ax.legend(fontsize=7, loc="lower left", markerscale=2)
plt.tight_layout(); plt.show()

Among the famous, talent and looks are **negatively** correlated — "why are talented
celebrities so often unattractive?" — even though they are unrelated in general. We
manufactured the correlation by conditioning on fame.


This is **collider bias** (*Berkson's paradox*) — not the fork behind *selection bias*.


Conditioning on a **descendant** of the collider works too. Follower count is caused by
fame; select on it instead of fame itself:

In [ ]:
#| echo: true
followers = 3 * famous + rng.normal(0, 1, n)       # a noisy proxy for fame
sel = followers > 2
print(f"corr(talent, looks | followers > 2) = {np.corrcoef(talent[sel], looks[sel])[0, 1]:+.2f}"
      f"   ({famous[sel].mean():.0%} of the selected are famous)")

A noisy proxy for a collider is still a collider: it opens the path *partially*.


## Mediators: don't "control away" your effect

If a job-training program raises earnings *by* teaching skills:

$$
\text{Training} \rightarrow \text{Skills} \rightarrow \text{Earnings}
$$

Controlling for **skills** would block the very pathway the program works through, making the
program look useless. This is **overcontrol bias**.


Rule of thumb for a **total** effect: adjust for common *causes*, never for anything on the
causal path or for common *effects* [@cinelli2020crash].


# Estimating effects

## Estimating effects under "no unmeasured confounders"

Suppose we've drawn the DAG and found an adjustment set $S$ that satisfies the backdoor
criterion. In potential-outcomes language this is **conditional ignorability**:

$$
\{Y(0), Y(1)\} \perp\!\!\!\perp T \mid S
$$

— randomization's independence, now holding only *within* levels of $S$. The DAG is how we
argue it; it cannot be checked from data.

From here on the treatment is $T$, as in the tutoring table; $X$ was convenient while
tracing paths.


## Four ways to adjust for $S$

Once $S$ is chosen, any of these recovers the effect:

1. **Regression adjustment** — include $S$ as covariates; read off the coefficient on $T$
   (this assumes the linear form is right: the regression is an estimator, not a licence).
2. **Stratification / standardization** — estimate the effect within levels of $S$, then
   average with the **population's** stratum shares (→ ATE); use the *treated* group's
   shares and you get the ATT.
3. **Matching** — pair each treated unit with similar control unit(s) on $S$.
4. **Propensity scores** — match or weight on $e(S) = P(T{=}1 \mid S)$, a one-number summary.


Identification (which $S$) and estimation (which estimator) are separate decisions. A
*wrong* $S$ gives a wrong answer with a tight standard error — no estimator rescues a bad graph.


## Propensity scores in one slide

When $S$ is high-dimensional, matching directly is hard. Rosenbaum and Rubin showed it is
enough to match or weight on the scalar **propensity score** $e(S) = P(T{=}1\mid S)$
[@rosenbaum1983central].

* **Estimate it** with any classifier of $T$ from $S$ — logistic regression (Lecture 11A)
  is the usual choice.
* **Match** each treated unit to controls with a similar $e$, or **weight** each unit by
  $1/e(S)$ if treated and $1/(1 - e(S))$ if not (inverse-propensity weighting, IPW).
* Weighting builds a "pseudo-population" in which treatment is unrelated to $S$ —
  mimicking a randomized experiment from observational data.


**Check overlap (positivity) first**

Plot the histogram of $e$ for treated and for control units. Where one group has units and
the other has none, there is no comparable counterpart and the estimate is extrapolation.
An $e$ of $0.01$ gives a weight of $100$: a tight standard error around a number that means
nothing.


## When confounders are *unobserved*: natural experiments

Adjustment only works for confounders we can **measure**. When we can't, we look for
"nature's randomization" — sources of variation in treatment that are *as good as random*
[@angrist2014mastering].

* **Difference-in-differences (DiD)** — compare the *change* over time in a treated group
  with the change in a control group; assumes **parallel trends**.
* **Regression discontinuity (RD)** — treatment switches on at a cutoff; compare units just
  above and just below it; assumes **nothing else jumps at the cutoff**.
* **Instrumental variables (IV)** — a variable that nudges the treatment as good as at
  random (next slide).


## An instrument, in graph terms

In [ ]:
#| echo: false
#| fig-align: center
#| label: iv-dag
iv_pos = {"L\nlottery": (0, 0), "T": (1, 0), "Y": (2, 0), "U\nunobserved": (1.5, 1)}
fig, ax = plt.subplots(figsize=(6.4, 2.8))
draw(ax, [("L\nlottery", "T"), ("T", "Y")], iv_pos, node_size=2000, font_size=9)
U = nx.DiGraph([("U\nunobserved", "T"), ("U\nunobserved", "Y")])
nx.draw_networkx_nodes(U, iv_pos, ax=ax, nodelist=["U\nunobserved"], node_color="white",
                       node_size=2900, edgecolors="#999", linewidths=1.2)
nx.draw_networkx_labels(U, iv_pos, ax=ax, labels={"U\nunobserved": "U\nunobserved"},
                        font_size=9, font_color="#777")
nx.draw_networkx_edges(U, iv_pos, ax=ax, node_size=2900, style="dashed",
                       edge_color="#999", arrowsize=22)
nx.draw_networkx_edges(nx.DiGraph([("L\nlottery", "Y")]), iv_pos, ax=ax, node_size=2000,
                       style="dashed", edge_color="#c0392b", width=2, arrowsize=22,
                       connectionstyle="arc3,rad=0.3")
ax.text(1, -0.55, "✗", color="#c0392b", fontsize=22, ha="center", va="center",
        bbox=dict(boxstyle="circle,pad=0.1", fc="white", ec="none"))
ax.text(1, -0.8, "no direct L → Y (exclusion)", color="#c0392b", fontsize=9,
        ha="center", va="top")
ax.set_xlim(-0.4, 2.4); ax.set_ylim(-1.1, 1.35)
plt.tight_layout(); plt.show()

1. **Relevance** — the arrow $L \to T$ exists (the lottery really changes who serves).
2. **Exclusion** — no arrow $L \to Y$: the lottery affects earnings *only through* service.
3. **Independence** — no backdoor path from $L$ to $Y$: nothing points into $L$ (it is random).


The unobserved $U$ confounds $T$ and $Y$, so adjustment is impossible — but $L$ is untouched
by $U$, and that is all IV needs.


## Assumptions you can't skip

Every observational causal estimate relies on assumptions [@hernan2020whatif]:

* **Conditional ignorability** — $\{Y(0),Y(1)\} \perp\!\!\!\perp T \mid S$: the adjustment set
  blocks every backdoor path. The DAG argues it; data cannot check it.
* **Positivity (overlap)** — every unit has a nonzero chance of each treatment ($0 < e(S) < 1$).
* **Correct graph** — the DAG reflects reality.


Since ignorability is untestable, good practice includes a **sensitivity analysis**:
how strong would an unmeasured confounder have to be to overturn the conclusion?


# Summary

## Summary

1. **A regression coefficient is an association, not an effect.** OLS estimates
   $P(Y \mid X)$; a causal claim is about $P(Y \mid do(X))$, and the two differ whenever
   there is confounding.
2. **A confounder is a common cause of treatment and outcome.** Adjusting for it (holding it
   fixed, or adding it to the regression) removes the spurious association — but only if you
   measured it and knew to include it.
3. **Potential outcomes $Y(1), Y(0)$ define the effect; you only ever observe one.** This is
   the fundamental problem of causal inference, so we estimate averages like the
   $\text{ATE} = \mathbb{E}[Y(1) - Y(0)]$.
4. **Naive difference in means $=$ causal effect $+$ selection bias.** Simpson's paradox is
   what selection bias looks like in a table: within-group and overall comparisons can point
   in opposite directions. Look for the confounder's two fingerprints; **direct
   standardization** gives every arm the same case mix.
5. **Randomization makes treatment independent of potential outcomes**, so the selection-bias
   term is zero, $\text{ATT} = \text{ATE}$, and the naive difference is unbiased for the ATE.
6. **A causal DAG encodes your assumptions, not the data's.** Chains and forks are open until
   you condition on them; colliders are closed until you do (or condition on a descendant).
7. **The backdoor procedure sorts every path into causal, backdoor or neither**, then blocks
   every open backdoor with non-descendants of $X$ — never a mediator, never a collider.
8. **When a confounder is unmeasured, adjustment cannot help.** Look for a natural experiment
   (DiD, RD, IV), state its untestable assumption, and report a sensitivity analysis.


## References

- [*Causal Inference: What If*](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/) — Hernán & Robins (free PDF)
- [*Causal Inference: A Primer*](http://bayes.cs.ucla.edu/PRIMER/) — Pearl, Glymour & Jewell
- [*The Book of Why*](http://bayes.cs.ucla.edu/WHY/) — Pearl & Mackenzie (popular intro)
- [*Causal Inference: The Mixtape*](https://mixtape.scunning.com/) — Cunningham (free online)
- [*Mastering 'Metrics*](https://www.masteringmetrics.com/) — Angrist & Pischke
- ["A Crash Course in Good and Bad Controls"](https://ftp.cs.ucla.edu/pub/stat_ser/r493.pdf) — Cinelli, Forney & Pearl
- [DoWhy documentation](https://www.pywhy.org/dowhy/) and [dagitty.net](https://dagitty.net/) — software for the draw → identify → estimate → refute workflow

Going deeper (optional): [Causal Inference: Estimating Effects](M14-Causal-Inference-Estimation.qmd)
works the estimators on simulated data (regression, stratification, IPW code, overlap),
covers DiD, IV and RD in detail, and ends with practice scenarios.

## Bibliography
